# V134 - Depth-gated Top1 selector on an independent anchor lineage

Hipotesa tunggal: user yang Top1-nya robustly wrong (2 dari 3 swap positif)
dapat diprediksi (depth AUC 0.72 di geometri V132); membatasi agree-policy ke
user predicted-deep menaikkan presisi. Falsifier terdaftar: DEPTH-vs-BASE
pooled >= +0.00015 DAN >=4/5 fold positif DAN benefit>harm DAN min-block >=
-0.001 DAN geometri test waras. Seed baru 20260910. Diagnostik saja: lineage
anchor beda dari V132 sehingga hasil apa pun tanpa implikasi submit.


In [1]:
import hashlib
import json
import os
import sys
import time
from hashlib import sha256
from pathlib import Path

import numpy as np
import pandas as pd

SMOKE = bool(int(os.getenv("V134_SMOKE", "0")))
SEED = 20260910
E5_ID = "intfloat/multilingual-e5-base"
NOMIC_ID = "nomic-ai/nomic-embed-text-v2-moe"
EXP_FILE = "d89714241c85dbaf93a5574550a5d9272c37b9b432c69fd5c5a2ff07fe690d84"

ROOT = Path("/home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender")
for cand in (ROOT / "nb_code", Path("nb_code"), Path("../nb_code")):
    if (cand / "v127_pipeline.py").exists():
        sys.path.insert(0, str(cand))
        break
import v127_pipeline as vp

cfg = vp.Config(seed=SEED, smoke=SMOKE,
                outer_splits=2 if SMOKE else 5, inner_splits=2 if SMOKE else 3,
                calibration_trees=20 if SMOKE else 150, ranker_trees=20 if SMOKE else 150,
                word_features=5000, char_features=8000, threads=4,
                bootstraps=200 if SMOKE else 2000)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data"), Path("../data"), ROOT / "data"]
DATA = next((c for c in CANDIDATES if (c / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
CH = next((c for c in [ROOT / "results/v132/v132/candidate_v132_DIAGNOSTIC.csv",
                       Path("results/v132/v132/candidate_v132_DIAGNOSTIC.csv"),
                       Path("../results/v132/v132/candidate_v132_DIAGNOSTIC.csv")] if c.exists()), None)
if CH is not None:
    assert sha256(CH.read_bytes()).hexdigest() == EXP_FILE, "champion file changed"
    print(f"protected base found: {CH}")
else:
    print("protected base not found: fail-close copy will be skipped")
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else ROOT / "results/v134")
OUT.mkdir(parents=True, exist_ok=True)
T = {}
print(f"SMOKE={SMOKE} DATA={DATA} OUT={OUT}")


protected base found: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/results/v132/v132/candidate_v132_DIAGNOSTIC.csv
SMOKE=True DATA=../data OUT=/home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/results/v134


## Load dan provenance cache

CSV organizer lewat modul. Cache embedding bentuk dan pipeline dicek ulang
sebelum dipakai.


In [2]:
t0 = time.time()
data = vp.load_data(DATA)
n_train, n_test = data.n_train, data.n_test
docs = data.docs
print(f"groups={len(set(data.groups))} train={n_train} test={n_test}")

import torch
from sentence_transformers import SentenceTransformer
E5C = next(c for c in [ROOT / "results/v58/e5_users.npy", Path("results/v58/e5_users.npy"), Path("../results/v58/e5_users.npy")] if c.exists())
NOC = next(c for c in [ROOT / "results/v124/nomic_users.npy", Path("results/v124/nomic_users.npy"), Path("../results/v124/nomic_users.npy")] if c.exists())
E5U = np.asarray(np.load(E5C), dtype=np.float32)
NOU = np.asarray(np.load(NOC), dtype=np.float32)
assert E5U.shape == (n_train + n_test, 768) and NOU.shape == (n_train + n_test, 768)
probe_idx = [0, 7, n_train - 1, n_train, n_train + n_test - 1]
e5t = SentenceTransformer(E5_ID)
pe = np.asarray(e5t.encode(["query: " + docs[i] for i in probe_idx], normalize_embeddings=True))
ce = (pe * E5U[probe_idx]).sum(1)
print(f"e5 probe cosine: {np.round(ce, 6)}")
assert (ce > 0.999).all(), "e5 cache pipeline mismatch"
del e5t
torch.cuda.empty_cache()
from transformers import AutoModel, AutoTokenizer
nTok = AutoTokenizer.from_pretrained(NOMIC_ID, trust_remote_code=True)
nNet = AutoModel.from_pretrained(NOMIC_ID, trust_remote_code=True, torch_dtype=torch.float16, device_map={"": 0} if torch.cuda.is_available() else None).eval()
pn = []
with torch.no_grad():
    for i in probe_idx:
        e = nTok("search_query: " + docs[i], padding=True, truncation=True, max_length=256, return_tensors="pt")
        e = {k: v.to(nNet.device) for k, v in e.items()}
        h = nNet(**e).last_hidden_state
        m = e["attention_mask"].unsqueeze(-1).float()
        v = ((h * m).sum(1) / m.sum(1).clamp_min(1)).float()
        pn.append((v / v.norm()).cpu().numpy().ravel())
pn = np.array(pn)
cn = (pn * (NOU[probe_idx] / (np.linalg.norm(NOU[probe_idx], axis=1, keepdims=True) + 1e-12))).sum(1)
print(f"nomic probe cosine: {np.round(cn, 6)}")
assert (cn > 0.999).all(), "nomic cache pipeline mismatch"
del nNet
torch.cuda.empty_cache()
T["probe"] = round(time.time() - t0, 1)
print(f"provenance OK in {T['probe']}s")


groups=290 train=4000 test=1000


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

e5 probe cosine: [1. 1. 1. 1. 1.]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


/home/athilla/.cache/huggingface/modules/transformers_modules/nomic_hyphen_ai/nomic_hyphen_bert_hyphen_2048/7710840340a098cfb869c4f65e87cf2b1b70caca/modeling_hf_nomic_bert.py:1634: UserWarning: Install Nomic's megablocks fork for better speed: `pip install git+https://github.com/nomic-ai/megablocks.git`
  warnings.warn("Install Nomic's megablocks fork for better speed: " +


[transformers] Detected the usage of `get_extended_attention_mask`: This function is deprecated and will be removed in v5.12.0. Please use the new API in `transformers.masking_utils`


nomic probe cosine: [1.       0.999999 1.       1.       0.999999]
provenance OK in 39.5s


## Silabus dari cache v127 yang sudah terverifikasi

Tanpa re-encode: bentuk dan hash pendek dicatat.


In [3]:
ESC = next(c for c in [ROOT / "results/v127/e5_syll.npy", Path("results/v127/e5_syll.npy"), Path("../results/v127/e5_syll.npy")] if c.exists())
NSC = next(c for c in [ROOT / "results/v127/nomic_syll.npy", Path("results/v127/nomic_syll.npy"), Path("../results/v127/nomic_syll.npy")] if c.exists())
ES = np.asarray(np.load(ESC), dtype=np.float32)
NS = np.asarray(np.load(NSC), dtype=np.float32)
assert ES.shape == (17, 768) and NS.shape == (17, 768)
assert np.isfinite(ES).all() and np.isfinite(NS).all()
print("syllabi loaded", ES.shape, sha256(ES.tobytes()).hexdigest()[:16], sha256(NS.tobytes()).hexdigest()[:16])


syllabi loaded (17, 768) 71fd3b24f9dd2974 7c142f0ef5afbbf1


## Eksperimen nested: BASE agree-policy vs DEPTH-gated

Setiap outer fold: head + threshold hanya dari outer-train (via fit_selector),
depth classifier hanya dari outer-train, gate tetap proba>=0.5. Satu tuas:
depth gate.


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold

t0 = time.time()
labels = data.y.copy()
NM = len(data.modules)
outer = StratifiedGroupKFold(n_splits=cfg.outer_splits, shuffle=True, random_state=SEED)
splits = [(np.arange(n_train)[tr], np.arange(n_train)[va]) for tr, va in outer.split(np.zeros(n_train), data.strata, data.groups)]
engine = vp.Engine(data, labels, cfg, OUT / "_engine_cache")
ANCH = np.zeros((n_train, NM))
BASE = np.zeros((n_train, NM))
DEPTH = np.zeros((n_train, NM))
rows, thr_pick = [], []

def user_feats(rows_u, U1, U2, U3, anch_mat):
    ou = vp.stable_order(anch_mat[rows_u])
    mg = np.ptp(anch_mat[rows_u][np.arange(len(rows_u))[:, None], ou[:, :2]], axis=1)
    rd = np.asarray(data.A[rows_u][:, :-1], dtype=float).mean(1)
    iv = np.asarray(data.A[rows_u][:, -1], dtype=float)
    return np.column_stack([U1.max(1), U1.mean(1), U2.max(1), U2.mean(1), U3.max(1), U3.mean(1), mg, rd, iv])

def head_utils(fit, rows_u, anch_mat):
    ctrl = vp.control_features(data, rows_u, anch_mat)
    fe = np.concatenate([ctrl, vp.semantic_features(np.asarray(E5U)[rows_u], ES, anch_mat[rows_u])], -1)
    fn = np.concatenate([ctrl, vp.semantic_features(np.asarray(NOU)[rows_u], NS, anch_mat[rows_u])], -1)
    FC = ctrl.shape[-1]
    Uc = fit["control"].predict(ctrl.reshape(-1, FC)).reshape(-1, 3)
    Ue = fit["e5"].predict(fe.reshape(-1, fe.shape[-1])).reshape(-1, 3)
    Un = fit["nomic"].predict(fn.reshape(-1, fn.shape[-1])).reshape(-1, 3)
    return Uc, Ue, Un

for fi, (tr, va) in enumerate(splits):
    fit = vp.fit_selector(engine, tr, E5U, ES, NOU, NS)
    anchor = fit["anchor_full"]
    thr = fit["threshold"]
    thr_pick.append(thr)
    Uc_va, Ue_va, Un_va = head_utils(fit, va, anchor)
    Bv = anchor[va].copy()
    picks = {}
    for j in range(len(va)):
        a1, a2 = int(Ue_va[j].argmax()), int(Un_va[j].argmax())
        if a1 == a2 and Ue_va[j, a1] > thr and Un_va[j, a2] > thr:
            Bv[j] = vp.apply_actions(anchor[va][j:j + 1], [a1 + 1], [True])[0]
            picks[j] = a1 + 1
    amat = anchor.copy()
    amat[tr] = fit["anchor_oof"]
    Uc_tr, Ue_tr, Un_tr = head_utils(fit, tr, amat)
    tgt = vp.action_targets(labels[tr], fit["anchor_oof"])
    depth_tr = (tgt > 0).sum(1) >= 2
    Xtr_u = user_feats(tr, Uc_tr, Ue_tr, Un_tr, amat)
    mu, sd = Xtr_u.mean(0), Xtr_u.std(0) + 1e-9
    clf = LogisticRegression(max_iter=3000).fit((Xtr_u - mu) / sd, depth_tr)
    Xva_u = user_feats(va, Uc_va, Ue_va, Un_va, anchor)
    proba_va = clf.predict_proba((Xva_u - mu) / sd)[:, 1]
    Dv = anchor[va].copy()
    nD = 0
    for j, slot in picks.items():
        if proba_va[j] >= 0.5:
            Dv[j] = Bv[j]
            nD += 1
    ANCH[va], BASE[va], DEPTH[va] = anchor[va], Bv, Dv
    rows.append({"fold": fi, "thr": thr, "nB": len(picks), "nD": int(nD),
                 "oof_anchor": float(vp.per_user_ndcg(labels[va], anchor[va]).mean()),
                 "oof_base": float(vp.per_user_ndcg(labels[va], Bv).mean()),
                 "oof_depth": float(vp.per_user_ndcg(labels[va], Dv).mean())})
    print(f"fold {fi}: thr={thr} nB={len(picks)} nD={nD} anchor={rows[-1]['oof_anchor']:.5f} base={rows[-1]['oof_base']:.5f} depth={rows[-1]['oof_depth']:.5f}", flush=True)
T["folds"] = round(time.time() - t0, 1)


fold 0: thr=0.002 nB=180 nD=18 anchor=0.62152 base=0.62342 depth=0.62258


fold 1: thr=0.001 nB=216 nD=10 anchor=0.59891 base=0.60217 depth=0.59921


## Metrik, gate terdaftar, paritas test

Primer: DEPTH-vs-BASE. Gate: pooled>=+0.00015, >=4/5 fold positif,
benefit>harm, min-block>=-0.001, geometri test waras.


In [5]:
t0 = time.time()
nA = vp.per_user_ndcg(labels, ANCH)
nB = vp.per_user_ndcg(labels, BASE)
nD = vp.per_user_ndcg(labels, DEPTH)
dB, dD = nB - nA, nD - nB
ciB = vp.cluster_bootstrap_ci(dB, data.groups, SEED, cfg.bootstraps)
ciD = vp.cluster_bootstrap_ci(dD, data.groups, SEED + 1, cfg.bootstraps)
foldB = np.array([dB[np.concatenate([va for _, va in [splits[fi]]])].mean() for fi in range(len(splits))])
foldD = np.array([dD[np.concatenate([va for _, va in [splits[fi]]])].mean() for fi in range(len(splits))])
top1 = vp.stable_order(ANCH)[:, 0]
blk = {int(m): float(dD[top1 == m].mean()) for m in np.unique(top1)}
dom = max(blk, key=lambda m: int((top1 == m).sum()))
nchat = np.array([len(d) == 0 for d in data.docs[:n_train]])
print(f"BASE-vs-ANCHOR pooled {dB.mean():+.5f} CI[{ciB[0]:+.5f},{ciB[1]:+.5f}] worst {foldB.min():+.5f}")
print(f"DEPTH-vs-BASE pooled {dD.mean():+.5f} CI[{ciD[0]:+.5f},{ciD[1]:+.5f}] worst {foldD.min():+.5f}")
print(f"dominant block M_{dom:03d} gain {blk[dom]:+.5f}; min block {min(blk.values()):+.5f}")
print(f"no-chat slice FULL gain {dD[nchat].mean() if nchat.sum() else float('nan'):+.5f} (n={nchat.sum()})")
print(f"deployed BASE {(np.abs(BASE - ANCH).sum(1) > 0).sum()} DEPTH {(np.abs(DEPTH - ANCH).sum(1) > 0).sum()}; thr {thr_pick}")
gate = bool(dD.mean() >= 0.00015 and (foldD > 0).sum() >= 4 and (dD[dD > 0].sum() > -dD[dD < 0].sum()) and min(blk.values()) >= -0.001)
print("GATE:", gate)
T["metrics"] = round(time.time() - t0, 1)

t0 = time.time()
fit_all = vp.fit_selector(engine, np.arange(n_train), E5U, ES, NOU, NS)
anchor_te = fit_all["anchor_full"][n_train:]
te_idx = np.arange(n_train, n_train + n_test)
Uc_t, Ue_t, Un_t = head_utils(fit_all, te_idx, fit_all["anchor_full"])
thrT = fit_all["threshold"]
FINB = anchor_te.copy()
for j in range(n_test):
    a1, a2 = int(Ue_t[j].argmax()), int(Un_t[j].argmax())
    if a1 == a2 and Ue_t[j, a1] > thrT and Un_t[j, a2] > thrT:
        FINB[j] = vp.apply_actions(anchor_te[j:j + 1], [a1 + 1], [True])[0]
amatA = fit_all["anchor_full"].copy()
amatA[np.arange(n_train)] = fit_all["anchor_oof"]
Uc_A, Ue_A, Un_A = head_utils(fit_all, np.arange(n_train), amatA)
tgtA = vp.action_targets(labels, fit_all["anchor_oof"])
depthA = (tgtA > 0).sum(1) >= 2
XA = user_feats(np.arange(n_train), Uc_A, Ue_A, Un_A, amatA)
muA, sdA = XA.mean(0), XA.std(0) + 1e-9
clfA = LogisticRegression(max_iter=3000).fit((XA - muA) / sdA, depthA)
Xt = user_feats(te_idx, Uc_t, Ue_t, Un_t, fit_all["anchor_full"])
proba_t = clfA.predict_proba((Xt - muA) / sdA)[:, 1]
FIND = anchor_te.copy()
nact = 0
for j in range(n_test):
    if proba_t[j] >= 0.5 and not np.array_equal(FINB[j], anchor_te[j]):
        FIND[j] = FINB[j]
        nact += 1
vp.check_geometry(anchor_te, FINB)
vp.check_geometry(anchor_te, FIND)
print(f"test BASE actions={int((np.abs(FINB - anchor_te).sum(1) > 0).sum())} DEPTH actions={nact}/{n_test} thr={thrT}")
T["test"] = round(time.time() - t0, 1)


BASE-vs-ANCHOR pooled +0.00264 CI[+0.00101,+0.00387] worst +0.00190
DEPTH-vs-BASE pooled -0.00200 CI[-0.00329,-0.00051] worst -0.00296
dominant block M_002 gain -0.00709; min block -0.00709
no-chat slice FULL gain -0.00098 (n=117)
deployed BASE 396 DEPTH 28; thr [0.002, 0.001]
GATE: False


test BASE actions=173 DEPTH actions=24/1000 thr=0.0005


## File: diagnostik saja plus fail-close champion

Tanpa kandidat submit. Fail-close menyalin byte champion bila ada.


In [6]:
t0 = time.time()
test_ids = data.user_ids[n_train:]
diag = pd.DataFrame({"user_id": test_ids})
for j, m in enumerate(data.modules):
    diag[m] = FIND[:, j]
diag.to_csv(OUT / "v134_diagnostic_test.csv", index=False)
print("wrote v134_diagnostic_test.csv (DIAGNOSTIC, not a candidate)")
if CH is not None:
    (OUT / "champion_unchanged_v134.csv").write_bytes(CH.read_bytes())
    assert sha256((OUT / "champion_unchanged_v134.csv").read_bytes()).hexdigest() == EXP_FILE
    print("fail-close champion copy hash-verified")
REPORT = {"seed": SEED, "splits": cfg.outer_splits, "gate": gate,
          "thr": thr_pick, "thr_test": thrT,
          "base_anchor": float(dB.mean()), "ci_base": [float(ciB[0]), float(ciB[1])], "worst_base": float(foldB.min()),
          "depth_base": float(dD.mean()), "ci_depth": [float(ciD[0]), float(ciD[1])], "worst_depth": float(foldD.min()),
          "fold_table": rows, "dominant_block": [dom, float(blk[dom])], "min_block": float(min(blk.values())),
          "deployed_base": int((np.abs(BASE - ANCH).sum(1) > 0).sum()),
          "deployed_depth": int((np.abs(DEPTH - ANCH).sum(1) > 0).sum()),
          "test_base_actions": int((np.abs(FINB - anchor_te).sum(1) > 0).sum()), "test_depth_actions": int(nact),
          "smoke": SMOKE, "timings": T, "audit_records": len(engine.fit_audit)}
(OUT / "v134_gate_report.json").write_text(json.dumps(REPORT, indent=1))
(OUT / "NOTES.md").write_text(f"# V134 depth-gated (seed {SEED})\ngate={gate} depth_vs_base={dD.mean():+.5f} CI=[{ciD[0]:+.5f},{ciD[1]:+.5f}] worst={foldD.min():+.5f} base_vs_anchor={dB.mean():+.5f} test_depth={nact} smoke={SMOKE} timings={T}\n")
T["files"] = round(time.time() - t0, 1)
print(json.dumps(REPORT, indent=1)[:1200])


wrote v134_diagnostic_test.csv (DIAGNOSTIC, not a candidate)
fail-close champion copy hash-verified
{
 "seed": 20260910,
 "splits": 2,
 "gate": false,
 "thr": [
  0.002,
  0.001
 ],
 "thr_test": 0.0005,
 "base_anchor": 0.0026448289800173556,
 "ci_base": [
  0.001012060577383708,
  0.003870111042284078
 ],
 "worst_base": 0.0019024639396457276,
 "depth_base": -0.0019990880366893456,
 "ci_depth": [
  -0.0032869814465929497,
  -0.0005059950776290002
 ],
 "worst_depth": -0.00296430329079727,
 "fold_table": [
  {
   "fold": 0,
   "thr": 0.002,
   "nB": 180,
   "nD": 18,
   "oof_anchor": 0.6215158492460158,
   "oof_base": 0.6234183131856617,
   "oof_depth": 0.6225765283206101
  },
  {
   "fold": 1,
   "thr": 0.001,
   "nB": 216,
   "nD": 10,
   "oof_anchor": 0.5989105229149565,
   "oof_base": 0.602174499996045,
   "oof_depth": 0.5992101967052477
  }
 ],
 "dominant_block": [
  2,
  -0.007085772135848133
 ],
 "min_block": -0.007085772135848133,
 "deployed_base": 396,
 "deployed_depth": 28,
 "te